<a href="https://colab.research.google.com/github/Maneesh290318/CourseRegistration/blob/main/Courseworkingwithopenai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Real MCP + OpenAI GPT Agentic Course Enrollment Chatbot

This Colab notebook builds an intelligent course-enrollment chatbot with a **real MCP server** and an **OpenAI-hosted GPT LLM**.

## Final architecture

User prompt  
→ Memory Agent  
→ Master Orchestrator Agent using OpenAI GPT  
→ SQL Agent / Analytics Agent / RAG Agent  
→ Real MCP Server Tools  
→ SQLite + DuckDB + ChromaDB  
→ Recommendation Agent using OpenAI GPT  
→ Response Agent using OpenAI GPT  
→ Gradio chatbot UI

## What this notebook proves

- A real MCP server is written to `mcp_server.py`.
- The notebook connects to the MCP server using an MCP stdio client.
- OpenAI GPT is used for orchestration, SQL generation, recommendation, and final response generation.
- SQLite stores courses, users, memory, chat history, and lead state.
- DuckDB stores analytics data loaded from Excel.
- ChromaDB stores curriculum and project knowledge for semantic search.
- LangGraph controls the multi-agent flow.
- Gradio allows live testing with multiple users.

In [ ]:
# import os
# from getpass import getpass
# from openai import OpenAI

# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

# openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# print("OpenAI client is ready.")

## Cell 1: Install required libraries

### Purpose
This cell installs all packages needed for the project.

### Why these packages are used
- `mcp`: creates and connects to the real MCP server.
- `openai`: calls OpenAI-hosted models.
- `duckdb`: runs analytics SQL over enrollment data.
- `chromadb`: stores vector embeddings for RAG.
- `sentence-transformers`: creates local embeddings for ChromaDB.
- `langgraph`: orchestrates the agent workflow.
- `gradio`: provides the testing UI.
- `pandas` and `openpyxl`: generate and read Excel data.
- `nest_asyncio`: allows async MCP calls inside Colab notebooks.

In [ ]:
!pip install -q mcp openai duckdb chromadb sentence-transformers langgraph gradio pandas openpyxl nest_asyncio
print("All required libraries installed.")

All required libraries installed.


In [ ]:
import os
import re
import sys
import json
import uuid
import random
import sqlite3
import asyncio
import textwrap
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, TypedDict
from getpass import getpass

import duckdb
import pandas as pd
import nest_asyncio
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from openai import OpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, END
import gradio as gr

nest_asyncio.apply()
random.seed(42)

# Setup OpenAI API key and client (moved from f7d12364)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client is ready.")

print("Imports completed successfully.")

OpenAI client is ready.
Imports completed successfully.


## Cell 2: Import libraries

### Purpose
This cell imports Python libraries used throughout the notebook.

In [ ]:
import os
import re
import sys
import json
import uuid
import random
import sqlite3
import asyncio
import textwrap
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, TypedDict
from getpass import getpass

import duckdb
import pandas as pd
import nest_asyncio
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, END
import gradio as gr

nest_asyncio.apply()
random.seed(42)

print("Imports completed successfully.")

Imports completed successfully.


## Cell 3: Global configuration

### Purpose
This cell defines file paths, model names, and project constants.

### Important OpenAI model note
This notebook uses `gpt-4o-mini` by default because it is fast and cost-efficient. You can change it to a more powerful model like `gpt-4o` for stronger reasoning if your OpenAI account allows it.

In [ ]:
BASE_DIR = Path.cwd()
SQLITE_DB = str(BASE_DIR / "phase2_courses_memory.db")
DUCKDB_DB = str(BASE_DIR / "phase2_enrollment_analytics.duckdb")
EXCEL_FILE = str(BASE_DIR / "synthetic_enrollment_records.xlsx")
CHROMA_PATH = str(BASE_DIR / "phase2_chroma_db")
MCP_SERVER_FILE = str(BASE_DIR / "mcp_server.py")

OPENAI_MODEL = "gpt-4o-mini"
DEFAULT_USER_ID = "U001"
DEFAULT_SESSION_ID = "S001"

print("Configuration loaded.")
print("OpenAI model:", OPENAI_MODEL)
print("SQLite DB:", SQLITE_DB)
print("DuckDB DB:", DUCKDB_DB)
print("Chroma path:", CHROMA_PATH)

Configuration loaded.
OpenAI model: gpt-4o-mini
SQLite DB: /content/phase2_courses_memory.db
DuckDB DB: /content/phase2_enrollment_analytics.duckdb
Chroma path: /content/phase2_chroma_db


## Cell 4: Add OpenAI API key

### Purpose
OpenAI requires an API key to call GPT models.

### Prompt used later by the LLM
The LLM receives role-specific prompts for routing, SQL generation, recommendation, and final response writing.

### How to get a key
Create an OpenAI API key, then paste it when this cell asks for it. The key is stored only in the notebook runtime environment.

In [ ]:
# if not os.environ.get("GROQ_API_KEY"):
#     os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

# groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
# print("Groq client is ready.")

## Cell 5: OpenAI helper function

### Purpose
This helper sends system and user prompts to OpenAI GPT and returns clean text.

### Why this is needed
All agents use the same LLM interface, so the notebook stays consistent and easy to debug.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

# Ensure OPENAI_MODEL is defined for call_openai's scope
# The value 'gpt-4o-mini' is copied from the global config in cell 647c6fee
OPENAI_MODEL = "gpt-4o-mini"

# Setup OpenAI API key and client directly in this cell to ensure availability
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client is ready within call_openai's cell.")

def call_openai(system_prompt: str, user_prompt: str, temperature: float = 0.2, max_tokens: int = 800) -> str:
    """Call OpenAI GPT and return the assistant response text."""
    # No need for 'global' keywords here as openai_client and OPENAI_MODEL are defined in this cell's scope.
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()

print("OpenAI helper function created.")

OpenAI client is ready within call_openai's cell.
OpenAI helper function created.


In [ ]:
print('--- Testing call_openai function ---')
sample_system_prompt = "You are a helpful assistant."
sample_user_prompt = "What is the capital of France?"

try:
    openai_response = call_openai(sample_system_prompt, sample_user_prompt)
    print("OpenAI Response:", openai_response)
except NameError as e:
    print(f"Error: {e}. Make sure openai_client and OPENAI_MODEL are defined.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Testing call_openai function ---
OpenAI Response: The capital of France is Paris.


## Cell 6: Create SQLite database

### Purpose
SQLite stores operational data:
- users
- sessions
- course catalog
- conversation history
- user interests
- lead status

### Output
The output confirms that all SQLite tables were created.

In [ ]:
def create_sqlite_database() -> None:
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()

    cur.executescript("""
    DROP TABLE IF EXISTS users;
    DROP TABLE IF EXISTS sessions;
    DROP TABLE IF EXISTS conversation_history;
    DROP TABLE IF EXISTS courses;
    DROP TABLE IF EXISTS user_interests;
    DROP TABLE IF EXISTS lead_status;

    CREATE TABLE users (
        user_id TEXT PRIMARY KEY,
        name TEXT,
        email TEXT,
        goal TEXT,
        experience_level TEXT,
        created_at TEXT
    );

    CREATE TABLE sessions (
        session_id TEXT PRIMARY KEY,
        user_id TEXT,
        started_at TEXT,
        last_active_at TEXT
    );

    CREATE TABLE conversation_history (
        message_id TEXT PRIMARY KEY,
        user_id TEXT,
        session_id TEXT,
        role TEXT,
        message TEXT,
        created_at TEXT
    );

    CREATE TABLE courses (
        course_id TEXT PRIMARY KEY,
        course_name TEXT,
        category TEXT,
        fee_inr INTEGER,
        duration TEXT,
        trainer TEXT,
        prerequisites TEXT,
        level TEXT,
        mode TEXT,
        syllabus TEXT
    );

    CREATE TABLE user_interests (
        interest_id TEXT PRIMARY KEY,
        user_id TEXT,
        interest TEXT,
        confidence REAL,
        updated_at TEXT
    );

    CREATE TABLE lead_status (
        lead_id TEXT PRIMARY KEY,
        user_id TEXT,
        status TEXT,
        score INTEGER,
        notes TEXT,
        updated_at TEXT
    );
    """)

    now = datetime.now().isoformat(timespec="seconds")
    cur.execute("INSERT INTO users VALUES (?, ?, ?, ?, ?, ?)", (DEFAULT_USER_ID, "Demo User", "demo@example.com", "Learn AI and get job-ready", "Beginner", now))
    cur.execute("INSERT INTO sessions VALUES (?, ?, ?, ?)", (DEFAULT_SESSION_ID, DEFAULT_USER_ID, now, now))
    cur.execute("INSERT INTO lead_status VALUES (?, ?, ?, ?, ?, ?)", ("L001", DEFAULT_USER_ID, "new", 20, "Initial website visitor", now))

    conn.commit()
    conn.close()

create_sqlite_database()
print("SQLite database and memory tables created successfully.")

SQLite database and memory tables created successfully.


## Cell 7: Insert 40 synthetic courses

### Purpose
This creates a realistic course catalog for the chatbot to query.

### Output
The output confirms that 40 courses were inserted into SQLite.

In [ ]:
COURSES = [('C001', 'Python for AI Foundations', 'Programming', 6999, '4 weeks', 'Ravi Kumar', 'Basic computer knowledge', 'Beginner', 'Online', 'Python for AI Foundations covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C002', 'SQL for Data Analytics', 'Data', 7999, '6 weeks', 'Meera Sharma', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'SQL for Data Analytics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C003', 'Machine Learning Professional', 'AI', 14999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Machine Learning Professional covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C004', 'Deep Learning with TensorFlow', 'AI', 15999, '10 weeks', 'Vikram Singh', 'Basic computer knowledge', 'Beginner', 'Online', 'Deep Learning with TensorFlow covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C005', 'Generative AI and LLM Apps', 'AI', 12999, '12 weeks', 'Neha Iyer', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Generative AI and LLM Apps covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C006', 'Agentic AI with LangGraph', 'AI', 9999, '4 weeks', 'Ravi Kumar', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Agentic AI with LangGraph covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C007', 'AWS Cloud Practitioner', 'Cloud', 8999, '6 weeks', 'Meera Sharma', 'Basic computer knowledge', 'Beginner', 'Online', 'AWS Cloud Practitioner covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C008', 'CI/CD with GitHub Actions', 'DevOps', 7999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'CI/CD with GitHub Actions covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C009', 'Cybersecurity Foundations', 'Security', 11999, '10 weeks', 'Vikram Singh', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Cybersecurity Foundations covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C010', 'Power BI Dashboarding', 'Data', 9999, '12 weeks', 'Neha Iyer', 'Basic computer knowledge', 'Beginner', 'Online', 'Power BI Dashboarding covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C011', 'RAG Chatbot Development', 'AI', 6999, '4 weeks', 'Ravi Kumar', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'RAG Chatbot Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C012', 'Computer Vision Bootcamp', 'AI', 7999, '6 weeks', 'Meera Sharma', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Computer Vision Bootcamp covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C013', 'Full Stack Web Development', 'Web', 14999, '8 weeks', 'Ananya Rao', 'Basic computer knowledge', 'Beginner', 'Online', 'Full Stack Web Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C014', 'Flutter App Development', 'Mobile', 15999, '10 weeks', 'Vikram Singh', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Flutter App Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C015', 'Product Management Basics', 'Business', 12999, '12 weeks', 'Neha Iyer', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Product Management Basics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C016', 'Advanced Excel Analytics', 'Data', 9999, '4 weeks', 'Ravi Kumar', 'Basic computer knowledge', 'Beginner', 'Online', 'Advanced Excel Analytics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C017', 'NLP with Transformers', 'AI', 8999, '6 weeks', 'Meera Sharma', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'NLP with Transformers covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C018', 'MLOps Deployment', 'AI', 7999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Advanced', 'Online', 'MLOps Deployment covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C019', 'Prompt Engineering Masterclass', 'AI', 11999, '10 weeks', 'Vikram Singh', 'Basic computer knowledge', 'Beginner', 'Online', 'Prompt Engineering Masterclass covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C020', 'Digital Marketing Analytics', 'Marketing', 9999, '12 weeks', 'Neha Iyer', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Digital Marketing Analytics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C021', 'Azure AI Fundamentals', 'Cloud', 6999, '4 weeks', 'Ravi Kumar', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Azure AI Fundamentals covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C022', 'Docker and Kubernetes', 'DevOps', 7999, '6 weeks', 'Meera Sharma', 'Basic computer knowledge', 'Beginner', 'Online', 'Docker and Kubernetes covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C023', 'Ethical Hacking Basics', 'Security', 14999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Ethical Hacking Basics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C024', 'Java Backend Development', 'Programming', 15999, '10 weeks', 'Vikram Singh', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Java Backend Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C025', 'Data Engineering with Spark', 'Data', 12999, '12 weeks', 'Neha Iyer', 'Basic computer knowledge', 'Beginner', 'Online', 'Data Engineering with Spark covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C026', 'AI Automation with Python', 'AI', 9999, '4 weeks', 'Ravi Kumar', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'AI Automation with Python covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C027', 'MCP Server Development', 'AI', 8999, '6 weeks', 'Meera Sharma', 'Python and SQL fundamentals', 'Advanced', 'Online', 'MCP Server Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C028', 'Software Testing Automation', 'Testing', 7999, '8 weeks', 'Ananya Rao', 'Basic computer knowledge', 'Beginner', 'Online', 'Software Testing Automation covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C029', 'Business Analytics', 'Business', 11999, '10 weeks', 'Vikram Singh', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Business Analytics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C030', 'UI/UX Design Foundations', 'Design', 9999, '12 weeks', 'Neha Iyer', 'Python and SQL fundamentals', 'Advanced', 'Online', 'UI/UX Design Foundations covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C031', 'LangChain Application Development', 'AI', 6999, '4 weeks', 'Ravi Kumar', 'Basic computer knowledge', 'Beginner', 'Online', 'LangChain Application Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C032', 'Voice AI Assistant Development', 'AI', 7999, '6 weeks', 'Meera Sharma', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'Voice AI Assistant Development covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C033', 'PostgreSQL for Developers', 'Database', 14999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Advanced', 'Online', 'PostgreSQL for Developers covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C034', 'Google Cloud Data Analytics', 'Cloud', 15999, '10 weeks', 'Vikram Singh', 'Basic computer knowledge', 'Beginner', 'Online', 'Google Cloud Data Analytics covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C035', 'C Programming Foundations', 'Programming', 12999, '12 weeks', 'Neha Iyer', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'C Programming Foundations covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C036', 'Statistics for Data Science', 'Data', 9999, '4 weeks', 'Ravi Kumar', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Statistics for Data Science covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C037', 'AI Career Accelerator', 'AI', 8999, '6 weeks', 'Meera Sharma', 'Basic computer knowledge', 'Beginner', 'Online', 'AI Career Accelerator covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C038', 'LLM Evaluation and Guardrails', 'AI', 7999, '8 weeks', 'Ananya Rao', 'Python and SQL fundamentals', 'Intermediate', 'Online', 'LLM Evaluation and Guardrails covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C039', 'Terraform Infrastructure as Code', 'DevOps', 11999, '10 weeks', 'Vikram Singh', 'Python and SQL fundamentals', 'Advanced', 'Online', 'Terraform Infrastructure as Code covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.'), ('C040', 'Interview Preparation for Tech Roles', 'Career', 9999, '12 weeks', 'Neha Iyer', 'Basic computer knowledge', 'Beginner', 'Online', 'Interview Preparation for Tech Roles covers practical concepts, hands-on labs, project work, interview preparation, and deployment-oriented learning.')]

def insert_courses() -> None:
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()
    cur.executemany("""
        INSERT INTO courses
        (course_id, course_name, category, fee_inr, duration, trainer, prerequisites, level, mode, syllabus)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, COURSES)
    conn.commit()
    conn.close()

insert_courses()
print(f"Inserted {len(COURSES)} courses into SQLite.")

Inserted 40 courses into SQLite.


## Cell 8: Generate enrollment Excel data

### Purpose
This creates synthetic enrollment records to simulate business analytics data.

### Why Excel is used
Many real business teams store lead and enrollment data in Excel before loading it into analytics tools.

In [ ]:
def generate_enrollment_excel(row_count: int = 50) -> pd.DataFrame:
    course_names = [course[1] for course in COURSES]
    lead_sources = ["Website", "Instagram", "YouTube", "Referral", "LinkedIn", "College Event"]
    payment_statuses = ["Paid", "Pending", "Partial", "Refunded"]
    batches = ["Batch-A", "Batch-B", "Batch-C", "Weekend", "Weekday"]

    rows = []
    start_date = datetime.now() - timedelta(days=90)
    for i in range(1, row_count + 1):
        course = random.choice(course_names)
        rows.append({
            "enrollment_id": f"E{i:03d}",
            "student_name": f"Student {i:02d}",
            "course_name": course,
            "lead_source": random.choice(lead_sources),
            "payment_status": random.choice(payment_statuses),
            "amount_paid": random.choice([0, 3000, 5000, 7999, 9999, 14999]),
            "batch": random.choice(batches),
            "enrolled_at": (start_date + timedelta(days=random.randint(0, 90))).date().isoformat(),
        })

    df = pd.DataFrame(rows)
    df.to_excel(EXCEL_FILE, index=False)
    return df

enrollment_df = generate_enrollment_excel()
print("Synthetic enrollment Excel generated:", EXCEL_FILE)
display(enrollment_df.head())

Synthetic enrollment Excel generated: /content/synthetic_enrollment_records.xlsx


,enrollment_id,student_name,course_name,lead_source,payment_status,amount_paid,batch,enrolled_at
0,E001,Student 01,CI/CD with GitHub Actions,Website,Partial,3000,Batch-B,2026-06-17
1,E002,Student 02,AWS Cloud Practitioner,College Event,Paid,9999,Weekend,2026-06-04
2,E003,Student 03,SQL for Data Analytics,Website,Pending,3000,Weekday,2026-08-16
3,E004,Student 04,SQL for Data Analytics,LinkedIn,Pending,14999,Weekday,2026-07-23
4,E005,Student 05,Product Management Basics,Referral,Partial,0,Batch-B,2026-08-28


## Cell 9: Load Excel data into DuckDB

### Purpose
DuckDB is used for analytical questions such as:
- Which course has the highest enrollment?
- Which lead source converts best?
- How much revenue came from paid enrollments?

In [ ]:
def create_duckdb_database() -> None:
    df = pd.read_excel(EXCEL_FILE)
    conn = duckdb.connect(DUCKDB_DB)
    conn.execute("DROP TABLE IF EXISTS enrollment_records")
    conn.register("enrollment_df", df)
    conn.execute("CREATE TABLE enrollment_records AS SELECT * FROM enrollment_df")
    conn.close()

create_duckdb_database()
print("DuckDB table enrollment_records created successfully.")

DuckDB table enrollment_records created successfully.


## Cell 10: Create ChromaDB vector database

### Purpose
ChromaDB stores unstructured knowledge such as curriculum chunks, project descriptions, and corporate training notes.

### Why this is needed
SQLite and DuckDB are best for structured data. ChromaDB is better for semantic search over text.

In [ ]:
def build_chroma_database() -> None:
    embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        client.delete_collection("course_knowledge")
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name="course_knowledge",
        embedding_function=embedding_function,
    )

    docs = [
        "Agentic AI with LangGraph covers agents, state graphs, memory, tools, routing, MCP integration, and deployment.",
        "Generative AI and LLM Apps covers prompt engineering, embeddings, RAG, vector databases, chatbot design, and evaluation.",
        "RAG Chatbot Development teaches document loading, chunking, embedding generation, retrieval, response synthesis, and hallucination control.",
        "MCP Server Development teaches tool registration, MCP client connection, server lifecycle, stdio transport, and safe tool execution.",
        "AI Career Accelerator includes resume building, interview preparation, portfolio projects, GitHub profile review, and mock interviews.",
        "For beginners, Python for AI Foundations is recommended before Machine Learning, RAG, LangChain, or Agentic AI.",
        "Corporate training can combine GenAI, RAG, Agentic AI, MCP, and MLOps into a 6-week implementation-focused program.",
        "A strong enrollment response should ask about learner goals, recommend a course path, explain benefits, and suggest a next step.",
    ]

    collection.add(
        ids=[f"doc_{i}" for i in range(len(docs))],
        documents=docs,
        metadatas=[{"source": "synthetic_curriculum"} for _ in docs],
    )

build_chroma_database()
print("ChromaDB course_knowledge collection created successfully.")

ChromaDB course_knowledge collection created successfully.


## Cell 11: Agent prompts

### Purpose
These are the system prompts that define the behavior of each agent.

### Prompt design principle
Each agent has one clear responsibility. This avoids one large prompt trying to do everything.

In [ ]:
AGENT_PROMPTS = {
    "master_orchestrator": """
You are the Master Orchestrator Agent for a course-enrollment chatbot.
Classify the user message into one of these routes:
- greeting
- course_lookup
- analytics
- rag_curriculum
- recommendation
- mixed
Return strict JSON with keys: intent, tools_needed, reason.
Do not answer the user directly.
""",
    "sql_agent": """
You are the SQL Agent. Convert the user's English question into a safe SQLite SELECT query.
Only use the courses table unless memory tables are explicitly needed.
Never generate INSERT, UPDATE, DELETE, DROP, ALTER, or PRAGMA.
Return only SQL, no markdown.
""",
    "analytics_agent": """
You are the Analytics Agent. Convert the user's English analytics question into a safe DuckDB SELECT query.
Use only the enrollment_records table.
Never generate INSERT, UPDATE, DELETE, DROP, ALTER, or PRAGMA.
Return only SQL, no markdown.
""",
    "recommendation_agent": """
You are the Recommendation Agent. Recommend courses based on the user goal, retrieved course rows, curriculum chunks, and memory.
Be practical, enrollment-focused, and honest about prerequisites.
""",
    "response_agent": """
You are the Response Agent for an education website chatbot.
Write a helpful, persuasive, concise final answer.
Use the retrieved facts. Do not invent course prices, durations, trainers, or syllabus details.
End with one clear next-step question.
""",
}

print("Agent prompts loaded:", list(AGENT_PROMPTS.keys()))

Agent prompts loaded: ['master_orchestrator', 'sql_agent', 'analytics_agent', 'recommendation_agent', 'response_agent']


## Cell 12: Write the real MCP server

### Purpose
This cell writes a real MCP server file named `mcp_server.py`.

### Why this is real MCP
The server uses `FastMCP` from the official MCP Python package and exposes tools through MCP stdio transport.

### MCP tools exposed
- `sqlite_schema`
- `sqlite_select`
- `duckdb_select`
- `chroma_search`
- `save_message`
- `load_memory`

In [ ]:
server_code = """from mcp.server.mcpserver import MCPServer
import sqlite3
import duckdb
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from datetime import datetime
import uuid
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("MCP_SERVER_FILE", "./mcp_server.py")).parent
SQLITE_DB = str(BASE_DIR / "phase2_courses_memory.db")
DUCKDB_DB = str(BASE_DIR / "phase2_enrollment_analytics.duckdb")
CHROMA_PATH = str(BASE_DIR / "phase2_chroma_db")

mcp = MCPServer("course-enrollment-real-mcp-server")


def _is_safe_select(sql: str) -> bool:
    lowered = sql.strip().lower()
    blocked = ["insert", "update", "delete", "drop", "alter", "create", "pragma", "attach", "detach", ";--"]
    return lowered.startswith("select") and not any(word in lowered for word in blocked)


@mcp.tool()
def sqlite_schema() -> str:
    \"\"\"Return SQLite table names and CREATE TABLE statements.\"\"\"\n    conn = sqlite3.connect(SQLITE_DB)
    rows = conn.execute(\"SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name\").fetchall()
    conn.close()\n    return \"\\n\\n\".join([f\"TABLE: {name}\\n{sql}\" for name, sql in rows])


@mcp.tool()
def sqlite_select(sql: str) -> str:\n    \"\"\"Run a safe read-only SELECT query on SQLite and return JSON records.\"\"\"\n    if not _is_safe_select(sql):\n        return \"ERROR: Only safe SELECT queries are allowed.\"\n    conn = sqlite3.connect(SQLITE_DB)\n    conn.row_factory = sqlite3.Row\n    rows = conn.execute(sql).fetchall()\n    conn.close()\n    return str([dict(row) for row in rows])


@mcp.tool()
def duckdb_select(sql: str) -> str:\n    \"\"\"Run a safe read-only SELECT query on DuckDB analytics data and return records.\"\"\"\n    if not _is_safe_select(sql):\n        return \"ERROR: Only safe SELECT queries are allowed.\"\n    conn = duckdb.connect(DUCKDB_DB, read_only=True)\n    df = conn.execute(sql).fetchdf()\n    conn.close()\n    return df.to_json(orient=\"records\")


@mcp.tool()
def chroma_search(query: str, n_results: int = 3) -> str:\n    \"\"\"Search the ChromaDB course knowledge collection semantically.\"\"\"\n    embedding_function = SentenceTransformerEmbeddingFunction(model_name=\"all-MiniLM-L6-v2\")\n    client = chromadb.PersistentClient(path=CHROMA_PATH)\n    collection = client.get_or_create_collection(name=\"course_knowledge\", embedding_function=embedding_function)\n    result = collection.query(query_texts=[query], n_results=n_results)\n    return str(result.get(\"documents\", [[]])[0])


@mcp.tool()
def save_message(user_id: str, session_id: str, role: str, message: str) -> str:\n    \"\"\"Save a chat message into SQLite conversation history.\"\"\"\n    conn = sqlite3.connect(SQLITE_DB)\n    conn.execute(\n        \"INSERT INTO conversation_history VALUES (?, ?, ?, ?, ?, ?)\",\n        (str(uuid.uuid4()), user_id, session_id, role, message, datetime.now().isoformat(timespec=\"seconds\")),\n    )\n    conn.commit()\n    conn.close()\n    return \"Message saved.\"\n

@mcp.tool()
def load_memory(user_id: str, session_id: str, limit: int = 6) -> str:\n    \"\"\"Load recent conversation history and lead status for a user/session.\"\"\"\n    conn = sqlite3.connect(SQLITE_DB)\n    conn.row_factory = sqlite3.Row\n    history = conn.execute(\n        \"\"\"\n        SELECT role, message, created_at\n        FROM conversation_history\n        WHERE user_id=? AND session_id=?\n        ORDER BY created_at DESC\n        LIMIT ?\n        \"\"\"\n,\n        (user_id, session_id, limit),\n    ).fetchall()\n    lead = conn.execute(\"SELECT status, score, notes FROM lead_status WHERE user_id=?\", (user_id,)).fetchall()\n    conn.close()\n    return str({\"recent_history\": [dict(row) for row in history], \"lead_status\": [dict(row) for row in lead]})\n

if __name__ == \"__main__\":\n    mcp.run()\n"""
Path(MCP_SERVER_FILE).write_text(server_code, encoding="utf-8")
print("Real MCP server file created:", MCP_SERVER_FILE)
print("First 20 lines of mcp_server.py:")
print("\n".join(Path(MCP_SERVER_FILE).read_text().splitlines()[:20]))

Real MCP server file created: /content/mcp_server.py
First 20 lines of mcp_server.py:
from mcp.server.mcpserver import MCPServer
import sqlite3
import duckdb
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from datetime import datetime
import uuid
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("MCP_SERVER_FILE", "./mcp_server.py")).parent
SQLITE_DB = str(BASE_DIR / "phase2_courses_memory.db")
DUCKDB_DB = str(BASE_DIR / "phase2_enrollment_analytics.duckdb")
CHROMA_PATH = str(BASE_DIR / "phase2_chroma_db")

mcp = MCPServer("course-enrollment-real-mcp-server")


def _is_safe_select(sql: str) -> bool:
    lowered = sql.strip().lower()


## Cell 13: MCP client helper

### Purpose
This cell creates a real MCP client that starts the server through stdio, initializes an MCP session, and calls MCP tools.

### Why this design is robust in Colab
The helper opens a fresh MCP session for each tool call. This is slower than a long-running connection but simpler and more reliable for notebooks.

In [ ]:
%%writefile mcp_call_helper.py
import asyncio
import json
import os
import sys
from typing import Any, Dict

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def _call_tool(tool_name: str, arguments: Dict[str, Any]) -> str:
    server_file = os.environ.get("MCP_SERVER_FILE", "mcp_server.py")
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[server_file],
        env=os.environ.copy(),
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            return "\n".join(
                content.text for content in result.content if hasattr(content, "text")
            )


def main() -> None:
    if len(sys.argv) != 3:
        raise SystemExit("Usage: python mcp_call_helper.py <tool_name> '<json_arguments>'")

    tool_name = sys.argv[1]
    arguments = json.loads(sys.argv[2])
    output = asyncio.run(_call_tool(tool_name, arguments))
    print(output)


if __name__ == "__main__":
    main()



Overwriting mcp_call_helper.py


In [ ]:
import subprocess

# Save the server file path for the helper subprocess.
os.environ["MCP_SERVER_FILE"] = MCP_SERVER_FILE


def call_mcp_tool(tool_name: str, arguments: Dict[str, Any]) -> str:
    """Call a real MCP tool from Colab/Jupyter safely.

    Why subprocess?
    Colab/Jupyter replaces standard input/output streams with notebook streams.
    The MCP stdio transport needs normal OS-level file descriptors, so direct
    calls from the notebook kernel can raise: UnsupportedOperation: fileno.
    Running the MCP client in a normal Python subprocess avoids that issue.
    """
    completed = subprocess.run(
        [sys.executable, "mcp_call_helper.py", tool_name, json.dumps(arguments)],
        text=True,
        capture_output=True,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            "MCP tool call failed.\n"
            f"Tool: {tool_name}\n"
            f"Arguments: {arguments}\n"
            f"STDOUT:\n{completed.stdout}\n"
            f"STDERR:\n{completed.stderr}"
        )

    return completed.stdout.strip()

print("Notebook-safe MCP client helper created.")



Notebook-safe MCP client helper created.


## Cell 14: Test real MCP server tools

### Purpose
This verifies that the notebook is not using a fake MCP-style class.

### Expected output
You should see SQLite table definitions and sample course rows returned through MCP.

In [ ]:
schema_text = call_mcp_tool("sqlite_schema", {})
print(schema_text[:1000])

sample_courses = call_mcp_tool("sqlite_select", {"sql": "SELECT course_id, course_name, fee_inr, duration FROM courses LIMIT 5"})
print("\nSample courses through MCP:")
print(sample_courses)

TABLE: conversation_history
CREATE TABLE conversation_history (
        message_id TEXT PRIMARY KEY,
        user_id TEXT,
        session_id TEXT,
        role TEXT,
        message TEXT,
        created_at TEXT
    )

TABLE: courses
CREATE TABLE courses (
        course_id TEXT PRIMARY KEY,
        course_name TEXT,
        category TEXT,
        fee_inr INTEGER,
        duration TEXT,
        trainer TEXT,
        prerequisites TEXT,
        level TEXT,
        mode TEXT,
        syllabus TEXT
    )

TABLE: lead_status
CREATE TABLE lead_status (
        lead_id TEXT PRIMARY KEY,
        user_id TEXT,
        status TEXT,
        score INTEGER,
        notes TEXT,
        updated_at TEXT
    )

TABLE: sessions
CREATE TABLE sessions (
        session_id TEXT PRIMARY KEY,
        user_id TEXT,
        started_at TEXT,
        last_active_at TEXT
    )

TABLE: user_interests
CREATE TABLE user_interests (
        interest_id TEXT PRIMARY KEY,
        user_id TEXT,
        interest TEXT,


## Cell 15: Safe SQL cleanup helpers

### Purpose
LLMs sometimes return markdown or explanations. These helpers extract only safe SQL before sending it to MCP.

In [ ]:
def clean_sql(text: str) -> str:
    text = text.strip()
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE).strip()
    # Modified to use re.DOTALL and simpler '.' instead of '[\s\S]' to avoid SyntaxWarning
    match = re.search(r"select.*", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(0).strip()
    text = text.rstrip(";")
    return text


def force_limit(sql: str, limit: int = 10) -> str:
    # Corrected regex for word boundary \b
    if re.search(r"\buni_table\b|\bunion\b|\blimit\b", sql, flags=re.IGNORECASE):
        return sql
    return f"{sql} LIMIT {limit}"

print("SQL cleanup helpers are ready.")

SQL cleanup helpers are ready.


## Cell 16: Agent state definition

### Purpose
LangGraph passes a shared state dictionary between agents. This typed state makes the workflow easier to understand.

In [ ]:
class ChatbotState(TypedDict, total=False):
    user_id: str
    session_id: str
    user_message: str
    memory: str
    route: Dict[str, Any]
    sqlite_sql: str
    duckdb_sql: str
    sqlite_result: str
    duckdb_result: str
    chroma_result: str
    recommendation: str
    final_response: str

print("ChatbotState defined.")

ChatbotState defined.


## Cell 17: Memory Agent

### Purpose
The Memory Agent stores the user message and loads recent session history.

### MCP tools used
- `save_message`
- `load_memory`

In [ ]:
def memory_agent(state: ChatbotState) -> ChatbotState:
    user_id = state.get("user_id", DEFAULT_USER_ID)
    session_id = state.get("session_id", DEFAULT_SESSION_ID)
    message = state["user_message"]

    call_mcp_tool("save_message", {
        "user_id": user_id,
        "session_id": session_id,
        "role": "user",
        "message": message,
    })
    memory = call_mcp_tool("load_memory", {
        "user_id": user_id,
        "session_id": session_id,
        "limit": 6,
    })
    state["memory"] = memory
    return state

print("Memory Agent created.")

Memory Agent created.


## Cell 18: Master Orchestrator Agent

### Purpose
The orchestrator decides which specialist agents should run.

### LLM prompt role
It returns JSON with the user intent and tools needed.

In [ ]:
def master_orchestrator_agent(state: ChatbotState) -> ChatbotState:
    prompt = f"""
User message: {state['user_message']}
Memory context: {state.get('memory', '')}

Return strict JSON only.
"""
    raw = call_openai(AGENT_PROMPTS["master_orchestrator"], prompt, temperature=0.0, max_tokens=300)
    try:
        route = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
    except Exception:
        route = {"intent": "mixed", "tools_needed": ["sqlite", "chroma"], "reason": "Fallback route because JSON parsing failed."}
    state["route"] = route
    return state

print("Master Orchestrator Agent created.")

Master Orchestrator Agent created.


## Cell 19: SQL Agent

### Purpose
The SQL Agent converts English into SQLite SQL and executes it through the MCP server.

### MCP tool used
- `sqlite_select`

In [ ]:
def sql_agent(state: ChatbotState) -> ChatbotState:
    route = state.get("route", {})
    tools_needed = " ".join(route.get("tools_needed", [])) + " " + route.get("intent", "")
    if not any(token in tools_needed.lower() for token in ["sqlite", "course", "recommendation", "mixed"]):
        state["sqlite_result"] = "[]"
        return state

    schema = call_mcp_tool("sqlite_schema", {})
    user_prompt = f"""
SQLite schema:
{schema}

User question:
{state['user_message']}

Generate one safe SELECT query using the courses table.
"""
    sql = clean_sql(call_openai(AGENT_PROMPTS["sql_agent"], user_prompt, temperature=0.0, max_tokens=300))
    sql = force_limit(sql, 10)
    state["sqlite_sql"] = sql
    state["sqlite_result"] = call_mcp_tool("sqlite_select", {"sql": sql})
    return state

print("SQL Agent created.")

SQL Agent created.


## Cell 20: Analytics Agent

### Purpose
The Analytics Agent converts English analytics questions into DuckDB SQL.

### MCP tool used
- `duckdb_select`

In [ ]:
def analytics_agent(state: ChatbotState) -> ChatbotState:
    route = state.get("route", {})
    route_text = " ".join(route.get("tools_needed", [])) + " " + route.get("intent", "") + " " + state["user_message"]
    if not any(token in route_text.lower() for token in ["analytics", "revenue", "enrollment", "lead source", "paid", "duckdb"]):
        state["duckdb_result"] = "[]"
        return state

    user_prompt = f"""
DuckDB table: enrollment_records
Columns: enrollment_id, student_name, course_name, lead_source, payment_status, amount_paid, batch, enrolled_at

User analytics question:
{state['user_message']}

Generate one safe SELECT query.
"""
    sql = clean_sql(call_openai(AGENT_PROMPTS["analytics_agent"], user_prompt, temperature=0.0, max_tokens=300))
    state["duckdb_sql"] = sql
    state["duckdb_result"] = call_mcp_tool("duckdb_select", {"sql": sql})
    return state

print("Analytics Agent created.")

Analytics Agent created.


## Cell 21: RAG Agent

### Purpose
The RAG Agent searches ChromaDB for curriculum, project, and corporate-training context.

### MCP tool used
- `chroma_search`

In [ ]:
def rag_agent(state: ChatbotState) -> ChatbotState:
    state["chroma_result"] = call_mcp_tool("chroma_search", {
        "query": state["user_message"],
        "n_results": 3,
    })
    return state

print("RAG Agent created.")

RAG Agent created.


## Cell 22: Recommendation Agent

### Purpose
This agent combines structured course rows, analytics output, vector search results, and memory into a course recommendation.

### LLM prompt behavior
It should recommend only what is supported by retrieved data.

In [ ]:
def recommendation_agent(state: ChatbotState) -> ChatbotState:
    user_prompt = f"""
User message:
{state['user_message']}

Memory:
{state.get('memory', '')}

SQLite course result:
{state.get('sqlite_result', '')}

DuckDB analytics result:
{state.get('duckdb_result', '')}

Chroma curriculum result:
{state.get('chroma_result', '')}

Create a practical recommendation for the learner.
"""
    state["recommendation"] = call_openai(AGENT_PROMPTS["recommendation_agent"], user_prompt, temperature=0.2, max_tokens=500)
    return state

print("Recommendation Agent created.")

Recommendation Agent created.


## Cell 23: Response Agent

### Purpose
The Response Agent writes the final user-facing answer and stores it in memory.

### MCP tool used
- `save_message`

In [ ]:
def response_agent(state: ChatbotState) -> ChatbotState:
    user_prompt = f"""
User message:
{state['user_message']}

Route:
{state.get('route', {})}

SQL query used:
{state.get('sqlite_sql', '')}

SQLite result:
{state.get('sqlite_result', '')}

DuckDB query used:
{state.get('duckdb_sql', '')}

DuckDB result:
{state.get('duckdb_result', '')}

Chroma result:
{state.get('chroma_result', '')}

Recommendation:
{state.get('recommendation', '')}

Write the final answer.
"""
    final = call_openai(AGENT_PROMPTS["response_agent"], user_prompt, temperature=0.3, max_tokens=700)
    state["final_response"] = final

    call_mcp_tool("save_message", {
        "user_id": state.get("user_id", DEFAULT_USER_ID),
        "session_id": state.get("session_id", DEFAULT_SESSION_ID),
        "role": "assistant",
        "message": final,
    })
    return state

print("Response Agent created.")

Response Agent created.


## Cell 24: Build LangGraph orchestration

### Purpose
LangGraph connects all agents into one flow.

### Flow
Memory → Orchestrator → SQL → Analytics → RAG → Recommendation → Response

In [ ]:
graph = StateGraph(ChatbotState)

graph.add_node("memory", memory_agent)
graph.add_node("orchestrator", master_orchestrator_agent)
graph.add_node("sql", sql_agent)
graph.add_node("analytics", analytics_agent)
graph.add_node("rag", rag_agent)
graph.add_node("recommendation", recommendation_agent)
graph.add_node("response", response_agent)

graph.set_entry_point("memory")
graph.add_edge("memory", "orchestrator")
graph.add_edge("orchestrator", "sql")
graph.add_edge("sql", "analytics")
graph.add_edge("analytics", "rag")
graph.add_edge("rag", "recommendation")
graph.add_edge("recommendation", "response")
graph.add_edge("response", END)

chatbot_app = graph.compile()
print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## Cell 25: Chat function

### Purpose
This function is used by both direct notebook testing and Gradio.

In [ ]:
def chat_with_agentic_bot(message: str, user_id: str = DEFAULT_USER_ID, session_id: Optional[str] = None) -> str:
    if not session_id:
        session_id = f"S-{user_id}"
    state: ChatbotState = {
        "user_id": user_id,
        "session_id": session_id,
        "user_message": message,
    }
    result = chatbot_app.invoke(state)
    return result["final_response"]

print("Chat function is ready.")

Chat function is ready.


## Cell 26: Test scenario 1 — beginner course recommendation

### User prompt
`I am a beginner and I want to learn AI. Which course should I start with?`

### Expected behavior
The bot should use SQLite course facts, ChromaDB curriculum context, and OpenAI GPT response generation.

In [ ]:
response_1 = chat_with_agentic_bot(
    "I am a beginner and I want to learn AI. Which course should I start with?",
    user_id="U001",
    session_id="S001",
)
print(response_1)

To start your journey in AI as a beginner, I recommend the "Python for AI Foundations" course. This course will equip you with essential programming skills in Python, which is crucial for AI development. After completing this course, you'll be well-prepared to explore more advanced topics like Machine Learning, RAG, LangChain, or Agentic AI.

Before diving into these AI-specific courses, ensure you have a basic understanding of programming concepts. Are you ready to enroll in the Python course?


## Cell 27: Test scenario 2 — analytics question

### User prompt
`Which courses have the highest enrollments and revenue?`

### Expected behavior
The bot should route to DuckDB analytics, then summarize the result in plain English.

In [ ]:
response_2 = chat_with_agentic_bot(
    "Which courses have the highest enrollments and revenue?",
    user_id="U002",
    session_id="S002",
)
print(response_2)

Based on our analysis, here are the top courses with the highest enrollments and revenue:

1. **C Programming Foundations** - 5 enrollments, $37,997 revenue
2. **Flutter App Development** - 4 enrollments, $35,998 revenue
3. **Product Management Basics** - 4 enrollments, $10,999 revenue
4. **Software Testing Automation** - 3 enrollments, $25,998 revenue
5. **AWS Cloud Practitioner** - 3 enrollments, $24,998 revenue

These courses are popular for a reason—they provide valuable skills that can enhance your career prospects. If you're interested in any of these courses, I recommend visiting the course page to learn more and enroll.

Which course would you like to explore further?


## Cell 28: Show stored conversation memory

### Purpose
This proves that session history is stored in SQLite through MCP.

In [ ]:
memory_check = call_mcp_tool("sqlite_select", {
    "sql": "SELECT user_id, session_id, role, message, created_at FROM conversation_history ORDER BY created_at DESC LIMIT 10"
})
print(memory_check)

ERROR: Only safe SELECT queries are allowed.


In [ ]:
#cell29

## Cell 29: Gradio chatbot UI

### Purpose
This launches a simple web interface to test the full agentic workflow.

### How to test
Try these prompts:
- `Hi, I want to learn AI from zero.`
- `What is the fee for Agentic AI with LangGraph?`
- `Which course is best for RAG chatbot development?`
- `Which lead source generated the most paid enrollments?`
- `Suggest a learning path for becoming a GenAI engineer.`

In [ ]:
def gradio_chat(user_message: str, user_id: str, history: List[List[str]]) -> str:
    session_id = f"GRADIO-{user_id or 'anonymous'}"
    return chat_with_agentic_bot(user_message, user_id=user_id or "anonymous", session_id=session_id)

with gr.Blocks(title="Real MCP + OpenAI GPT Course Enrollment Chatbot") as demo:
    gr.Markdown("# Real MCP + OpenAI GPT Course Enrollment Chatbot")
    gr.Markdown("Test SQLite, DuckDB, ChromaDB, MCP tools, LangGraph orchestration, and OpenAI GPT responses.")
    user_id_box = gr.Textbox(label="User ID", value="U001")
    chatbot = gr.ChatInterface(
        fn=lambda message, history: gradio_chat(message, user_id_box.value, history),
        chatbot=gr.Chatbot(height=420),
        textbox=gr.Textbox(placeholder="Ask about courses, fees, curriculum, or enrollment analytics..."),
    )

demo.launch(share=False, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.


## Cell 30: Streamlit chatbot UI

### Purpose
This launches a simple web interface using Streamlit to test the full agentic workflow as an alternative to Gradio.

### How to test
After running the Streamlit app, a URL will be provided. Open this URL in your browser. You can then use the provided prompts to test the bot:
- `Hi, I want to learn AI from zero.`
- `What is the fee for Agentic AI with LangGraph?`
- `Which course is best for RAG chatbot development?`
- `Which lead source generated the most paid enrollments?`
- `Suggest a learning path for becoming a GenAI engineer.`

In [ ]:
!pip install -q streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
changed 22 packages in 1s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [ ]:
streamlit_app_code = '''
import streamlit as st
import os
import re
import sys
import json
import uuid
import random
import sqlite3
import asyncio
import textwrap
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, TypedDict
from getpass import getpass

import duckdb
import pandas as pd
import nest_asyncio
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from openai import OpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.graph import StateGraph, END

nest_asyncio.apply()
random.seed(42)

# Global configuration - Copied from cell 647c6fee
BASE_DIR = Path.cwd()
SQLITE_DB = str(BASE_DIR / "phase2_courses_memory.db")
DUCKDB_DB = str(BASE_DIR / "phase2_enrollment_analytics.duckdb")
EXCEL_FILE = str(BASE_DIR / "synthetic_enrollment_records.xlsx")
CHROMA_PATH = str(BASE_DIR / "phase2_chroma_db")
MCP_SERVER_FILE = str(BASE_DIR / "mcp_server.py") # Ensure this path is correct relative to where streamlit_app.py runs

OPENAI_MODEL = "gpt-4o-mini"
DEFAULT_USER_ID = "U001"
DEFAULT_SESSION_ID = "S001"

# OpenAI client setup - Copied from cell f7d12364
# In Streamlit, environment variables are typically already set or can be passed.
# getpass() won't work in this context.
if not os.environ.get("OPENAI_API_KEY"): # This should ideally be set in the Colab env before launching Streamlit
    # Fallback for local testing if not set, but not ideal for deployed apps
    # For Colab, the key should be set in the previous cell and inherited.
    st.error("OPENAI_API_KEY environment variable not set. Please set it in your Colab environment.")
    st.stop()

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def call_openai(system_prompt: str, user_prompt: str, temperature: float = 0.2, max_tokens: int = 800) -> str:
    """Call OpenAI GPT and return the assistant response text."""
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()

# MCP client helper - Copied and adapted from sg3aJNgDjn09
# The mcp_call_helper.py must exist in the same directory
# and MCP_SERVER_FILE env var must be set correctly.

os.environ["MCP_SERVER_FILE"] = MCP_SERVER_FILE

def call_mcp_tool(tool_name: str, arguments: Dict[str, Any]) -> str:
    """Call a real MCP tool from Streamlit safely."""
    completed = subprocess.run(
        [sys.executable, "mcp_call_helper.py", tool_name, json.dumps(arguments)],
        text=True,
        capture_output=True,
        check=False,
    )

    if completed.returncode != 0:
        error_message = (
            "MCP tool call failed.\n"
            f"Tool: {tool_name}\n"
            f"Arguments: {arguments}\n"
            f"STDOUT:\n{completed.stdout}\n"
            f"STDERR:\n{completed.stderr}"
        )
        st.error(error_message)
        raise RuntimeError(error_message)

    return completed.stdout.strip()

# SQL cleanup helpers - Copied from cell 7a165808
def clean_sql(text: str) -> str:
    text = text.strip()
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE).strip()
    # Modified this line to use re.DOTALL and simpler '.' instead of '[\s\S]' to avoid SyntaxWarning
    match = re.search(r"select.*", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(0).strip()
    text = text.rstrip(";")
    return text


def force_limit(sql: str, limit: int = 10) -> str:
    # Corrected regex for word boundary \b
    if re.search(r"\buni_table\b|\bunion\b|\blimit\b", sql, flags=re.IGNORECASE):
        return sql
    return f"{sql} LIMIT {limit}"

# Agent state definition - Copied from cell c842382a
class ChatbotState(TypedDict, total=False):
    user_id: str
    session_id: str
    user_message: str
    memory: str
    route: Dict[str, Any]
    sqlite_sql: str
    duckdb_sql: str
    sqlite_result: str # Added missing field
    duckdb_result: str # Added missing field
    chroma_result: str
    recommendation: str
    final_response: str

# Agent prompts - Copied from cell 418211c3
AGENT_PROMPTS = {
    "master_orchestrator": """
You are the Master Orchestrator Agent for a course-enrollment chatbot.
Classify the user message into one of these routes:
- greeting
- course_lookup
- analytics
- rag_curriculum
- recommendation
- mixed
Return strict JSON with keys: intent, tools_needed, reason.
Do not answer the user directly.
""",
    "sql_agent": """
You are the SQL Agent. Convert the user's English question into a safe SQLite SELECT query.
Only use the courses table unless memory tables are explicitly needed.
Never generate INSERT, UPDATE, DELETE, DROP, ALTER, or PRAGMA.
Return only SQL, no markdown.
""",
    "analytics_agent": """
You are the Analytics Agent. Convert the user's English analytics question into a safe DuckDB SELECT query.
Use only the enrollment_records table.
Never generate INSERT, UPDATE, DELETE, DROP, ALTER, or PRAGMA.
Return only SQL, no markdown.
""",
    "recommendation_agent": """
You are the Recommendation Agent. Recommend courses based on the user goal, retrieved course rows, curriculum chunks, and memory.
Be practical, enrollment-focused, and honest about prerequisites.
""",
    "response_agent": """
You are the Response Agent for an education website chatbot.
Write a helpful, persuasive, concise final answer.
Use the retrieved facts. Do not invent course prices, durations, trainers, or syllabus details.
End with one clear next-step question.
""",
}

# Agent functions - Copied from their respective cells
def memory_agent(state: ChatbotState) -> ChatbotState:
    user_id = state.get("user_id", DEFAULT_USER_ID)
    session_id = state.get("session_id", DEFAULT_SESSION_ID)
    message = state["user_message"]

    call_mcp_tool("save_message", {
        "user_id": user_id,
        "session_id": session_id,
        "role": "user",
        "message": message,
    })
    memory = call_mcp_tool("load_memory", {
        "user_id": user_id,
        "session_id": session_id,
        "limit": 6,
    })
    state["memory"] = memory
    return state

def master_orchestrator_agent(state: ChatbotState) -> ChatbotState:
    prompt = f"""
User message: {state['user_message']}
Memory context: {state.get('memory', '')}

Return strict JSON only.
"""
    raw = call_openai(AGENT_PROMPTS["master_orchestrator"], prompt, temperature=0.0, max_tokens=300)
    try:
        route = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
    except Exception:
        route = {"intent": "mixed", "tools_needed": ["sqlite", "chroma"], "reason": "Fallback route because JSON parsing failed."}
    state["route"] = route
    return state

def sql_agent(state: ChatbotState) -> ChatbotState:
    route = state.get("route", {})
    tools_needed = " ".join(route.get("tools_needed", [])) + " " + route.get("intent", "")
    if not any(token in tools_needed.lower() for token in ["sqlite", "course", "recommendation", "mixed"]):
        state["sqlite_result"] = "[]"
        return state

    schema = call_mcp_tool("sqlite_schema", {})
    user_prompt = f"""
SQLite schema:
{schema}

User question:
{state['user_message']}

Generate one safe SELECT query using the courses table. \
"""
    sql = clean_sql(call_openai(AGENT_PROMPTS["sql_agent"], user_prompt, temperature=0.0, max_tokens=300))
    sql = force_limit(sql, 10)
    state["sqlite_sql"] = sql
    state["sqlite_result"] = call_mcp_tool("sqlite_select", {"sql": sql})
    return state

def analytics_agent(state: ChatbotState) -> ChatbotState:
    route = state.get("route", {})
    route_text = " ".join(route.get("tools_needed", [])) + " " + route.get("intent", "") + " " + state["user_message"]
    if not any(token in route_text.lower() for token in ["analytics", "revenue", "enrollment", "lead source", "paid", "duckdb"]):
        state["duckdb_result"] = "[]"
        return state

    user_prompt = f"""
DuckDB table: enrollment_records
Columns: enrollment_id, student_name, course_name, lead_source, payment_status, amount_paid, batch, enrolled_at

User analytics question:
{state['user_message']}

Generate one safe SELECT query.
"""
    sql = clean_sql(call_openai(AGENT_PROMPTS["analytics_agent"], user_prompt, temperature=0.0, max_tokens=300))
    state["duckdb_sql"] = sql
    state["duckdb_result"] = call_mcp_tool("duckdb_select", {"sql": sql})
    return state

def rag_agent(state: ChatbotState) -> ChatbotState:
    state["chroma_result"] = call_mcp_tool("chroma_search", {
        "query": state["user_message"],
        "n_results": 3,
    })
    return state

def recommendation_agent(state: ChatbotState) -> ChatbotState:
    user_prompt = f"""
User message:
{state['user_message']}

Memory:
{state.get('memory', '')}

SQLite course result:
{state.get('sqlite_result', '')}

DuckDB analytics result:
{state.get('duckdb_result', '')}

Chroma curriculum result:
{state.get('chroma_result', '')}

Create a practical recommendation for the learner.
"""
    state["recommendation"] = call_openai(AGENT_PROMPTS["recommendation_agent"], user_prompt, temperature=0.2, max_tokens=500)
    return state

def response_agent(state: ChatbotState) -> ChatbotState:
    user_prompt = f"""
User message:
{state['user_message']}

Route:
{state.get('route', {})}

SQL query used:
{state.get('sqlite_sql', '')}

SQLite result:
{state.get('sqlite_result', '')}

DuckDB query used:
{state.get('duckdb_sql', '')}

DuckDB result:
{state.get('duckdb_result', '')}

Chroma result:
{state.get('chroma_result', '')}

Recommendation:
{state.get('recommendation', '')}

Write the final answer.
"""
    final = call_openai(AGENT_PROMPTS["response_agent"], user_prompt, temperature=0.3, max_tokens=700)
    state["final_response"] = final

    call_mcp_tool("save_message", {
        "user_id": state.get("user_id", DEFAULT_USER_ID),
        "session_id": state.get("session_id", DEFAULT_SESSION_ID),
        "role": "assistant",
        "message": final,
    })
    return state

# LangGraph orchestration - Copied from cell 344b748e
graph = StateGraph(ChatbotState)

graph.add_node("memory", memory_agent)
graph.add_node("orchestrator", master_orchestrator_agent)
graph.add_node("sql", sql_agent)
graph.add_node("analytics", analytics_agent)
graph.add_node("rag", rag_agent)
graph.add_node("recommendation", recommendation_agent)
graph.add_node("response", response_agent)

graph.set_entry_point("memory")
graph.add_edge("memory", "orchestrator")
graph.add_edge("orchestrator", "sql")
graph.add_edge("sql", "analytics")
graph.add_edge("analytics", "rag")
graph.add_edge("rag", "recommendation")
graph.add_edge("recommendation", "response")
graph.add_edge("response", END)

chatbot_app = graph.compile()

# Chat function - Copied from cell 683979f3
def chat_with_agentic_bot(message: str, user_id: str = DEFAULT_USER_ID, session_id: Optional[str] = None) -> str:
    if not session_id:
        session_id = f"S-{user_id}"
    state: ChatbotState = {
        "user_id": user_id,
        "session_id": session_id,
        "user_message": message,
    }
    result = chatbot_app.invoke(state)
    return result["final_response"]


# Streamlit App Layout
st.title("Course Enrollment Chatbot (Streamlit)")
st.write("Test SQLite, DuckDB, ChromaDB, MCP tools, LangGraph orchestration, and OpenAI GPT responses.")

user_id = st.sidebar.text_input("User ID", value="U001")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# React to user input
if prompt := st.chat_input("Ask about courses, fees, curriculum, or enrollment analytics..."):
    # Display user message in chat message container
    st.chat_message("user").markdown(prompt)
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.spinner("Thinking..."):
        try:
            response = chat_with_agentic_bot(prompt, user_id=user_id)
        except Exception as e:
            response = f"Error: {e}"

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        st.markdown(response)
    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

import subprocess # Needed for subprocess.run in Streamlit app

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_app_code)

print("streamlit_app.py created with all necessary components.")

streamlit_app.py created with all necessary components.


<>:99: SyntaxWarning: invalid escape sequence '\s'
<>:99: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1433/392268600.py:99: SyntaxWarning: invalid escape sequence '\s'
  # Modified this line to use re.DOTALL and simpler '.' instead of '[\s\S]' to avoid SyntaxWarning


In [ ]:
# Run Streamlit app in the background and expose it via localtunnel
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS=False --server.enableXsrfProtection=False > streamlit.log 2>&1 &
!sleep 3
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://some-laws-wave.loca.lt
^C


## Cell 30: Final checklist

### What is included
- Real MCP server using `FastMCP`
- Real MCP client using stdio transport
- OpenAI GPT LLM calls
- SQLite operational database
- DuckDB analytics database
- ChromaDB vector search
- LangGraph orchestration
- Memory storage
- Gradio testing UI
- Clear prompts, comments, markdown descriptions, and expected outputs

### Submission note
Before submitting, run the notebook from the top in a fresh Colab runtime after adding your OpenAI API key.

In [ ]:
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt
from IPython.display import HTML # Import HTML for local display

# This cell serves to illustrate the architecture of the 'Real MCP + OpenAI GPT Agentic Course Enrollment Chatbot'
# through a Knowledge Graph. This visualization and accompanying explanation are crucial for all stakeholders:
#
# - **Layman/Business Folks**: Understand the high-level components and how they interact to deliver the chatbot's functionality and business value.
# - **Technical Folks/Developers**: Gain a clear, visual overview of the system's design, agent interactions, data flows, and tool usage, aiding in development, debugging, and maintenance.
#
# The output includes:
# 1. A detailed markdown explanation within the cell's output, covering the 'what', 'why', 'when', and 'how' of knowledge graphs in this project.
# 2. Python code to construct a graph using `networkx` based on the notebook's architecture.
# 3. Visualization of this graph using `pyvis`, generating an interactive HTML file for exploration.

# 1. Explanation of Knowledge Graphs in this Project Context
# This section is markdown as requested, explaining the what, why, and how of KGs.
print("""# Understanding the Project's Knowledge Graph\n\n## What is a Knowledge Graph?\nA Knowledge Graph (KG) is a structured representation of information that describes real-world entities and their relationships in a machine-readable format. It uses a graph-based data model where 'nodes' represent entities (e.g., courses, agents, databases) and 'edges' represent the relationships between them (e.g., 'uses', 'stores', 'routes_to').

## Why are Knowledge Graphs Used?\nKnowledge graphs provide a powerful way to organize, integrate, and query complex, heterogeneous data. They enable: \n- **Semantic understanding**: Machines can understand the meaning and context of data, not just keywords.\n- **Data integration**: Unifying disparate data sources by mapping them to a common conceptual model.\n- **Inference and reasoning**: Discovering new facts or relationships by traversing the graph.\n- **Explainability**: Making AI systems more transparent by showing the underlying data and logic used for decisions.\n\n## When to Use Knowledge Graphs?\nKGs are particularly useful when dealing with: \n- **Complex domains**: Where relationships between entities are crucial (like in an agentic system). \n- **Heterogeneous data**: Combining structured (SQL, DuckDB) and unstructured (ChromaDB) information.\n- **Need for explainability**: When understanding the 'why' behind a system's output is important.\n- **Evolving data**: KGs can be easily extended without re-architecting the entire system.\n\n## Benefits of Having a Knowledge Graph for this Project\nIn the context of the 'Real MCP + OpenAI GPT Agentic Course Enrollment Chatbot', a knowledge graph offers several significant benefits:\n\n1.  **Enhanced System Understanding and Debugging**: By visualizing the entire architecture (agents, databases, tools, LLMs), developers can quickly grasp how different components interact. This makes debugging easier, as the flow of information and control is clearly mapped.\n\n2.  **Improved Maintainability and Onboarding**: New developers joining the project can get up to speed much faster by consulting the knowledge graph. It serves as a living documentation of the system's structure and dependencies, reducing the learning curve.\n\n3.  **Facilitating Feature Expansion and Impact Analysis**: When planning new features or modifications, the KG helps identify which agents, tools, or data sources would be affected. This allows for more informed decision-making and minimizes unintended side effects.\n\n4.  **Optimized Agent Routing and Orchestration (Future Potential)**: While the current orchestrator uses LLM classification, a KG could potentially be used to inform more sophisticated routing logic. For example, if a query involves a specific course, the KG could highlight that the `SQL Agent` and `ChromaDB` are relevant, leading to more efficient tool selection.\n\n5.  **Better RAG Contextualization**: The RAG Agent could potentially leverage the KG to understand not just 'what' curriculum content is relevant, but 'how' it relates to other courses, prerequisites, or learning paths, providing richer context to the LLM.\n\n6.  **Data Governance and Lineage**: The KG can document where data originates (SQLite, DuckDB, ChromaDB), how it flows through agents, and where it's used, improving data governance and ensuring data integrity.\n\n7.  **Explaining Bot Decisions to Users (Future Potential)**: Although not directly implemented in the current response agent, a KG could serve as a backend for generating explanations to users about 'why' a particular course was recommended, by showing the path through relevant knowledge.\n## How the Knowledge Graph Helps Achieve Overall Objectives\nThe primary objective of this project is to build an intelligent, agentic course-enrollment chatbot. A knowledge graph directly supports this by:\n\n-   **Clarifying the Agentic Architecture**: It visually represents the interplay between the `Master Orchestrator`, `Memory Agent`, `SQL Agent`, `Analytics Agent`, `RAG Agent`, `Recommendation Agent`, and `Response Agent`, making the complex LangGraph flow intuitive.\n-   **Mapping Data Sources to Agents**: It clearly shows which agents interact with `SQLite` (courses, memory), `DuckDB` (analytics), and `ChromaDB` (curriculum knowledge), ensuring that data is accessed and utilized appropriately for different intents.\n-   **Highlighting Tool Utilization**: The graph can depict which MCP tools (`sqlite_schema`, `sqlite_select`, `duckdb_select`, `chroma_search`, `save_message`, `load_memory`) are used by which agents, providing a comprehensive overview of the system's capabilities.\n-   **Ensuring Cohesion and Consistency**: By having a unified view of the system, it helps ensure that all components work together cohesively towards providing accurate course information, relevant recommendations, and insightful analytics to the user.\n\nIn essence, the knowledge graph transforms the implicit understanding of the system's design into an explicit, shareable, and analyzable structure, directly contributing to the project's success by improving development, maintenance, and potential future enhancements.\n""")

# 2. Create the Knowledge Graph using NetworkX
# This section defines the nodes (entities) and edges (relationships) of the chatbot system.
# Each node is assigned a 'type' and 'color' for better visual categorization.
# Edges are given 'labels' to describe the interaction.

G = nx.DiGraph() # Initialize a directed graph

# Define Nodes with their types and colors
nodes = [
    ("User", {"type": "External", "color": "red"}),
    ("Gradio/Streamlit UI", {"type": "Interface", "color": "lightcoral"}),
    ("Chatbot App", {"type": "Core", "color": "blue"}),
    ("Memory Agent", {"type": "Agent", "color": "lightblue"}),
    ("Master Orchestrator Agent", {"type": "Agent", "color": "lightblue"}),
    ("SQL Agent", {"type": "Agent", "color": "lightblue"}),
    ("Analytics Agent", {"type": "Agent", "color": "lightblue"}),
    ("RAG Agent", {"type": "Agent", "color": "lightblue"}),
    ("Recommendation Agent", {"type": "Agent", "color": "lightblue"}),
    ("Response Agent", {"type": "Agent", "color": "lightblue"}),
    ("OpenAI GPT LLM", {"type": "LLM", "color": "lightgreen"}),
    ("MCP Server", {"type": "Tool Gateway", "color": "purple"}),
    ("SQLite DB (Courses, Memory)", {"type": "Database", "color": "orange"}),
    ("DuckDB DB (Analytics)", {"type": "Database", "color": "orange"}),
    ("ChromaDB (Vector DB)", {"type": "Database", "color": "orange"}),
    ("mcp_call_helper.py", {"type": "Helper", "color": "grey"})
]

G.add_nodes_from(nodes)

# Define Edges (Relationships) with labels describing the interaction
edges = [
    ("User", "Gradio/Streamlit UI", {"label": "interacts with"}),
    ("Gradio/Streamlit UI", "Chatbot App", {"label": "sends query to"}),
    ("Chatbot App", "Memory Agent", {"label": "invokes"}),
    ("Memory Agent", "MCP Server", {"label": "calls save/load tools"}),
    ("MCP Server", "SQLite DB (Courses, Memory)", {"label": "accesses (save/load)"}),
    ("Memory Agent", "Master Orchestrator Agent", {"label": "passes state to"}),
    ("Master Orchestrator Agent", "OpenAI GPT LLM", {"label": "routes via"}),
    ("Master Orchestrator Agent", "SQL Agent", {"label": "routes to (SQL intent)"}),
    ("Master Orchestrator Agent", "Analytics Agent", {"label": "routes to (Analytics intent)"}),
    ("Master Orchestrator Agent", "RAG Agent", {"label": "routes to (RAG intent)"}),
    ("SQL Agent", "OpenAI GPT LLM", {"label": "generates SQL via"}),
    ("SQL Agent", "MCP Server", {"label": "executes SQL via"}),
    ("MCP Server", "SQLite DB (Courses, Memory)", {"label": "queries"}),
    ("Analytics Agent", "OpenAI GPT LLM", {"label": "generates SQL via"}),
    ("Analytics Agent", "MCP Server", {"label": "executes SQL via"}),
    ("MCP Server", "DuckDB DB (Analytics)", {"label": "queries"}),
    ("RAG Agent", "MCP Server", {"label": "performs search via"}),
    ("MCP Server", "ChromaDB (Vector DB)", {"label": "searches"}),
    ("RAG Agent", "Recommendation Agent", {"label": "passes results to"}),
    ("SQL Agent", "Recommendation Agent", {"label": "passes results to"}),
    ("Analytics Agent", "Recommendation Agent", {"label": "passes results to"}),
    ("Recommendation Agent", "OpenAI GPT LLM", {"label": "formulates rec. via"}),
    ("Recommendation Agent", "Response Agent", {"label": "passes rec. to"}),
    ("Response Agent", "OpenAI GPT LLM", {"label": "generates final response via"}),
    ("Response Agent", "MCP Server", {"label": "calls save_message tool"}),
    ("MCP Server", "SQLite DB (Courses, Memory)", {"label": "saves history to"}),
    ("Response Agent", "Chatbot App", {"label": "returns response to"}),
    ("Chatbot App", "Gradio/Streamlit UI", {"label": "displays to"}),
    ("MCP Server", "mcp_call_helper.py", {"label": "runs via"}), # Simplified representation for clarity
    ("mcp_call_helper.py", "MCP Server", {"label": "communicates with"}) # Simplified representation for clarity
]

G.add_edges_from(edges)

# 3. Visualize the Knowledge Graph using Pyvis
# Pyvis creates an interactive visualization, allowing you to zoom, pan, and drag nodes.
# Node colors are based on their 'type' for easy differentiation.

net = Network(notebook=True, height="750px", width="100%", cdn_resources='remote', directed=True, bgcolor="#222222", font_color="white")

# Add nodes to the Pyvis network with customized properties
for node, attributes in G.nodes(data=True):
    net.add_node(node, label=node, title=node, group=attributes["type"], color=attributes["color"], physics=True, font={'size': 12})

# Add edges to the Pyvis network with customized properties
for source, target, attributes in G.edges(data=True):
    net.add_edge(source, target, title=attributes["label"], label=attributes["label"], width=1, arrows='to', physics=True, font={'size': 9, 'color': 'lightgray'})

# Configure physics and interaction options for better visualization
net.set_options("""
var options = {\n  "physics": {\n    "enabled": true,\n    "barnesHut": {\n      "gravitationalConstant": -2000,\n      "centralGravity": 0.3,\n      "springLength": 95,\n      "springConstant": 0.04,\n      "damping": 0.09,\n      "avoidOverlap": 0.5\n    },\n    "maxVelocity": 50,\n    "minVelocity": 0.1,\n    "solver": "barnesHut"\n  },\n  "interaction": {\n    "dragNodes": true,\n    "hover": true,\n    "tooltipDelay": 200\n  },\n  "edges": {\n    "color": {\n      "inherit": false,\n      "color": "lightgray"\n    }\n  }\n}\n""")

# Generate the HTML file for the interactive graph
output_filename = "project_knowledge_graph.html"
net.show(output_filename)

# Display the HTML content directly in the Colab output
with open(output_filename, 'r') as f:
    html_content = f.read()
display(HTML(html_content))

print(f"\nInteractive Knowledge Graph saved to {output_filename}. You can open this file in your browser to view the graph. This visual representation helps in understanding the complex interdependencies within the chatbot's architecture, making it easier to comprehend for both technical and non-technical audiences.\n")

# Explanation of the Output:
# The output first provides a comprehensive textual explanation of Knowledge Graphs
# and their specific relevance and benefits to this project. Following this, the Python
# code executes to create the graph. Finally, a message indicates that an interactive HTML
# file ('project_knowledge_graph.html') has been generated. This HTML file contains the
# visual knowledge graph, where:
# - Nodes (circles) represent different components of the system (e.g., Agents, Databases, LLMs).
# - Node colors differentiate between types of components (e.g., red for User, blue for Core/Agents, orange for Databases).
# - Edges (arrows) show the directed relationships and flow of information/control between components, with labels explaining the nature of the relationship.
# This interactive visualization is a powerful tool for understanding the system's design at a glance.

# Understanding the Project's Knowledge Graph

## What is a Knowledge Graph?
A Knowledge Graph (KG) is a structured representation of information that describes real-world entities and their relationships in a machine-readable format. It uses a graph-based data model where 'nodes' represent entities (e.g., courses, agents, databases) and 'edges' represent the relationships between them (e.g., 'uses', 'stores', 'routes_to').

## Why are Knowledge Graphs Used?
Knowledge graphs provide a powerful way to organize, integrate, and query complex, heterogeneous data. They enable: 
- **Semantic understanding**: Machines can understand the meaning and context of data, not just keywords.
- **Data integration**: Unifying disparate data sources by mapping them to a common conceptual model.
- **Inference and reasoning**: Discovering new facts or relationships by traversing the graph.
- **Explainability**: Making AI systems more transparent by showing the underlying data and logic used for decisions.


Interactive Knowledge Graph saved to project_knowledge_graph.html. You can open this file in your browser to view the graph. This visual representation helps in understanding the complex interdependencies within the chatbot's architecture, making it easier to comprehend for both technical and non-technical audiences.



In [ ]:
## Understanding the Chatbot's Agentic Architecture: A Deep Dive

This section explains, in detail, how the 'Real MCP + OpenAI GPT Agentic Course Enrollment Chatbot' is structured and implemented across the Colab notebook. We will break down the system into its core components – the agents, their interactions, the data stores, and the tools they utilize – providing insights for **layman, technical, and business audiences**.

Our goal is to demystify the complex interplay of these elements, showing how they work together to deliver a smart, responsive, and efficient course recommendation and information system. We'll walk through the entire flow, from a user's initial query to the bot's final response, highlighting specific code cells that bring this architecture to life.

### High-Level Overview (for Layman & Business Folks)

Imagine a team of specialized experts working together to answer your questions about courses. That's essentially what our chatbot is! When you ask a question, it doesn't just go to one giant brain. Instead, it goes through a carefully managed process involving several 'agents,' each with a specific job:

1.  **Memory Keeper**: This agent remembers what you've talked about before, so the chatbot can provide personalized responses.
2.  **Traffic Controller (Orchestrator)**: This is the boss. It listens to your question and decides which other expert agents need to be involved to answer it best (e.g., "Is this about course details, analytics, or just a general chat?").
3.  **Data Sleuths (SQL, Analytics, RAG Agents)**: These are the specialists. One looks up course details in a structured database (SQL Agent), another crunches numbers for enrollment trends (Analytics Agent), and a third searches through detailed curriculum documents (RAG Agent).
4.  **Course Advisor (Recommendation Agent)**: This agent takes all the information gathered by the data sleuths and your past interactions, then formulates the best course recommendations for you.
5.  **Communicator (Response Agent)**: This agent crafts the final, easy-to-understand answer and delivers it to you, always making sure to ask a follow-up question to keep the conversation going.

This modular design makes the chatbot very powerful: it can handle diverse queries, remember context, and easily be updated or expanded without breaking everything. It's like having a highly efficient, multi-talented team at your service 24/7.

Now, let's delve deeper into the technical implementation of each component, referencing the specific cells in this notebook.

### 1. User Input and Interface (`Gradio/Streamlit UI` & `Chatbot App`)

**Layman/Business**: This is where a user types their question into the chatbot's window (either a Gradio or Streamlit interface).

**Technical**: The user interacts with either the Gradio UI (defined in **Cell 30** with `gr.ChatInterface`) or the Streamlit UI (defined in **Cell 31** `streamlit_app_code` and launched in **Cell 32**).

When a user types a message, the UI calls the `chat_with_agentic_bot` function (**Cell 25**), passing the user's message, `user_id`, and `session_id`. This function initializes the `ChatbotState` (`ChatbotState` in **Cell 16**), which acts as a shared scratchpad for all agents to add or retrieve information during the conversation. This `ChatbotState` contains the `user_message` and user/session identifiers.

**Real-Time Scenario**: A user types: "Hi, I'm new here and want to learn about AI courses."
- `user_message`: "Hi, I'm new here and want to learn about AI courses."
- `user_id`: "U001" (or whatever is set in the UI).
- `session_id`: "GRADIO-U001" (generated dynamically).

### 2. Memory Agent (`memory_agent`)

**Layman/Business**: Just like a human customer service agent remembers your past conversations, the Memory Agent makes sure our chatbot remembers what you've said before. This helps it understand context and provide more relevant answers.

**Technical**: The `memory_agent` (**Cell 17**) is the first node in our LangGraph workflow (defined in **Cell 24**). Its responsibilities are two-fold:
1.  **Save Current Message**: It saves the `user_message` to the `conversation_history` table in the SQLite database using the `save_message` MCP tool. This ensures a persistent record of the interaction.
2.  **Load Past Memory**: It then loads recent conversation history and the user's `lead_status` from SQLite using the `load_memory` MCP tool. This memory is crucial for the Orchestrator and other agents to understand the conversation's context and user's profile.

The `ChatbotState` is updated with this retrieved `memory`.

**Real-Time Scenario**: After the user types "Hi, I'm new here and want to learn about AI courses."
- The `memory_agent` saves this message.
- It then retrieves previous interactions for `user_id='U001'` and `session_id='GRADIO-U001'`, and potentially the `lead_status` for `U001` (e.g., `{'recent_history': [...], 'lead_status': [{'status': 'new', 'score': 20}]}`).
- This `memory` is added to the `ChatbotState`.

### 3. Master Orchestrator Agent (`master_orchestrator_agent`)

**Layman/Business**: This is the 'brain' or 'traffic controller' of our chatbot. It reads your question and the conversation history, then decides which specialist agent (e.g., the 'course details' expert, the 'analytics' expert, or the 'curriculum search' expert) is best suited to handle your query. It's like a triage nurse directing you to the right doctor.

**Technical**: The `master_orchestrator_agent` (**Cell 18**) receives the `ChatbotState` (including the `user_message` and `memory`). It constructs a prompt using the `user_message` and the `memory` and sends it to the `call_openai` helper function (defined in **Cell 5**, `f7d12364`) along with the `master_orchestrator` system prompt from `AGENT_PROMPTS` (**Cell 21**).

OpenAI's GPT model then classifies the intent of the user's query into one of several predefined routes (e.g., `greeting`, `course_lookup`, `analytics`, `rag_curriculum`, `recommendation`, `mixed`). It also suggests `tools_needed`.

The `ChatbotState` is updated with the `route` (a dictionary containing `intent`, `tools_needed`, and `reason`). This `route` dictates which subsequent agents will be activated by LangGraph.

**Business**: This agent ensures efficiency by directing queries to the most appropriate resource, preventing unnecessary computations and speeding up response times. It's crucial for managing operational costs and maintaining high customer satisfaction.

**Real-Time Scenario**: Following our example, the user's message "Hi, I'm new here and want to learn about AI courses." combined with memory, goes to the Orchestrator.
- The OpenAI LLM might classify the intent as `course_lookup` or `recommendation`, indicating `tools_needed` like `sqlite` and `chroma`.
- The `ChatbotState` is updated with `route = {'intent': 'course_lookup', 'tools_needed': ['sqlite', 'chroma'], 'reason': 'User is asking for AI course information.'}`.

### 4. Agent Routing by Master Orchestrator

**Layman/Business**: Once the Master Orchestrator has figured out what you want (your 'intent') and what kind of tools might be needed, it doesn't try to answer the question itself. Instead, it acts like a conductor, telling the other specialist agents who should jump in next. For example, if you're asking for course details, it signals the 'SQL Agent' to query the course database. If it's about enrollment trends, it calls on the 'Analytics Agent'.

**Technical**: The routing logic isn't explicitly hardcoded with `if/else` statements for each intent in the `master_orchestrator_agent` function (`Cell 18`). Instead, the `master_orchestrator_agent`'s output, `state['route']`, which contains the `intent` (e.g., `course_lookup`, `analytics`, `rag_curriculum`) is consumed by the LangGraph framework (defined in `Cell 24`).

LangGraph uses this `route['intent']` to determine the *next node* in the graph. While the current `graph` definition in `Cell 24` shows a linear flow (`memory -> orchestrator -> sql -> analytics -> rag -> recommendation -> response`), a more sophisticated LangGraph setup would use conditional edges based on the `route['intent']`. In a more advanced implementation, the `Master Orchestrator` would return a decision that directly influences the next agent called. For this project, all agents run sequentially after the orchestrator, but the *intent* determined by the orchestrator still guides how each subsequent agent processes the `user_message` and potentially skips its own operations if its specific intent doesn't match.

For example, the `sql_agent` (`Cell 19`) explicitly checks the `route['intent']` and `tools_needed` to decide if it should generate and execute a SQL query. If the intent isn't `sqlite`, `course`, `recommendation`, or `mixed`, it might return an empty result, essentially 'skipping' its core operation.

**Business**: This routing mechanism is crucial for ensuring that the chatbot efficiently uses its resources. By identifying the user's intent early, it avoids unnecessary processing by agents that aren't relevant to the query, leading to faster response times and optimized computational costs.

### 5. Specialist Agents: Information Retrieval and Processing

Based on the Orchestrator's `route` (even in this sequential setup, the agents adapt their behavior), the specialist agents get to work. Each agent has a distinct role in gathering or processing information, often interacting with a specific data source via the MCP Server.

#### 5a. SQL Agent (`sql_agent`)

**Layman/Business**: This agent is our 'course catalog expert.' If you ask about specific courses, fees, or course schedules, this agent goes to the main course database to fetch precise details.

**Technical**: The `sql_agent` (`Cell 19`) checks the `ChatbotState['route']` to see if its services are required. If so, it first retrieves the SQLite database schema (`sqlite_schema` MCP tool) to understand the available tables and columns. It then uses the `call_openai` function (`Cell 5`, `f7d12364`) with the `sql_agent` system prompt (`Cell 21`) to generate a safe SQL `SELECT` query based on the `user_message`. This query is then cleaned (`clean_sql` and `force_limit` in `Cell 15`) and executed against the SQLite database using the `sqlite_select` MCP tool. The results are stored in `state['sqlite_result']`.

**Real-Time Scenario**: User asks: "What is the fee for Agentic AI with LangGraph?"
- Orchestrator routes to `course_lookup`.
- `sql_agent` gets the SQLite schema, generates `SELECT fee_inr FROM courses WHERE course_name LIKE '%Agentic AI with LangGraph%'`.
- Executes this via `sqlite_select` and stores `[{'fee_inr': 9999}]` in `state['sqlite_result']`.

#### 5b. Analytics Agent (`analytics_agent`)

**Layman/Business**: This agent is like our 'business analyst.' If you want to know about enrollment trends, popular courses, or revenue figures, this agent digs into our sales data.

**Technical**: The `analytics_agent` (`Cell 20`) also checks the `ChatbotState['route']` for keywords like `analytics`, `revenue`, `enrollment`, etc. If relevant, it uses `call_openai` with the `analytics_agent` system prompt (`Cell 21`) to generate a DuckDB `SELECT` query. This query is cleaned and then executed using the `duckdb_select` MCP tool. The JSON results are stored in `state['duckdb_result']`.

**Real-Time Scenario**: User asks: "Which courses have the highest enrollments and revenue?"
- Orchestrator routes to `analytics`.
- `analytics_agent` generates a DuckDB query like `SELECT course_name, COUNT(*) AS enrollments, SUM(amount_paid) AS total_revenue FROM enrollment_records GROUP BY course_name ORDER BY enrollments DESC, total_revenue DESC LIMIT 5`.
- Executes this via `duckdb_select` and stores the top 5 courses by enrollment/revenue in `state['duckdb_result']`.

#### 5c. RAG Agent (`rag_agent`)

**Layman/Business**: This is our 'curriculum researcher.' If your question is about course content, prerequisites, or broader AI concepts not easily found in simple database lookups, this agent searches through detailed course descriptions and knowledge documents.

**Technical**: The `rag_agent` (`Cell 21`) directly calls the `chroma_search` MCP tool with the `user_message` as the query. This tool performs a semantic search against the ChromaDB vector store (`ChromaDB` created in `Cell 10`) to retrieve relevant text chunks from the course knowledge base. These retrieved chunks provide rich, contextual information and are stored in `state['chroma_result']`.

**Real-Time Scenario**: User asks: "Suggest a learning path for becoming a GenAI engineer."
- Orchestrator routes to `rag_curriculum`.
- `rag_agent` searches ChromaDB for terms like 'GenAI engineer learning path' and retrieves documents like "Generative AI and LLM Apps covers prompt engineering..." and "Corporate training can combine GenAI, RAG, Agentic AI, MCP, and MLOps..." in `state['chroma_result']`.

### 6. Recommendation Agent (`recommendation_agent`)

**Layman/Business**: This agent is your 'personal course advisor.' After all the information has been gathered by the specialist agents, this agent combines everything – your question, past interactions, course details, analytics, and curriculum snippets – to give you a thoughtful and personalized course recommendation.

**Technical**: The `recommendation_agent` (`Cell 22`) gathers all the results stored in the `ChatbotState`: `user_message`, `memory`, `sqlite_result`, `duckdb_result`, and `chroma_result`. It then constructs a comprehensive prompt and sends it to `call_openai` using the `recommendation_agent` system prompt (`Cell 21`). The LLM processes all this information and generates a natural language recommendation, which is stored in `state['recommendation']`.

**Real-Time Scenario**: Combining results from our previous examples:
- `user_message`: "I am a beginner and I want to learn AI. Which course should I start with?"
- `memory`: Shows user is new.
- `sqlite_result`: Contains basic course info if course names were looked up.
- `chroma_result`: Contains relevant curriculum chunks like "For beginners, Python for AI Foundations is recommended...".
- The LLM synthesizes this to recommend "Python for AI Foundations" and explains why.

### 7. Response Agent (`response_agent`)

**Layman/Business**: Finally, this is the 'friendly communicator.' It takes the recommendation (or any other answer generated by the agents) and turns it into a clear, concise, and helpful message for you. It also makes sure to save its own response to remember for next time and always asks a follow-up question to keep the conversation flowing.

**Technical**: The `response_agent` (`Cell 23`) receives the full `ChatbotState`, including `user_message`, `route`, all SQL queries and their results, ChromaDB results, and the `recommendation`. It uses `call_openai` with the `response_agent` system prompt (`Cell 21`) to generate the `final_response` in natural language. After generating the response, it uses the `save_message` MCP tool to save its own message (role: 'assistant') to the conversation history. The `final_response` is then returned to the calling function, which ultimately displays it to the user.

**Real-Time Scenario**: Following the recommendation, the Response Agent:
- Crafts a message like: "To start your journey in AI as a beginner, I recommend the 'Python for AI Foundations' course... Are you ready to enroll in the Python course?"
- Saves this assistant message to SQLite.
- The chatbot UI displays this message to the user.

### Summary of Flow (from User to Response)

1.  **User Input**: User types a query into the Gradio/Streamlit UI (`Cell 30`/`Cell 31`).
2.  **`chat_with_agentic_bot`**: Function (`Cell 25`) receives the message and initializes the `ChatbotState` (`Cell 16`).
3.  **`memory_agent`**: Saves the user's message and loads past conversation history and lead status (`Cell 17`).
4.  **`master_orchestrator_agent`**: Uses an LLM to classify the user's intent and identify `tools_needed`, updating the `ChatbotState['route']` (`Cell 18`).
5.  **`sql_agent`**: If the `route` suggests a course lookup, it generates and executes SQLite SQL via MCP (`Cell 19`).
6.  **`analytics_agent`**: If the `route` suggests analytics, it generates and executes DuckDB SQL via MCP (`Cell 20`).
7.  **`rag_agent`**: Searches ChromaDB for relevant curriculum context via MCP (`Cell 21`).
8.  **`recommendation_agent`**: Combines all gathered information (user message, memory, SQL/analytics/RAG results) and uses an LLM to formulate a course recommendation (`Cell 22`).
9.  **`response_agent`**: Takes the recommendation, crafts a final user-facing message, saves it to memory via MCP, and returns it (`Cell 23`).
10. **UI Display**: The final response is shown to the user in the Gradio/Streamlit UI.

This entire flow is orchestrated by `LangGraph` (`Cell 24`), which defines the sequence of agent calls based on the defined graph structure, ensuring a coherent and intelligent conversation experience.

### How `ChatbotState` Maintains Data Consistency Across Agents

**Layman/Business**: Imagine a shared document that everyone on a team updates in real-time. Each team member (agent) adds their part to the document, and everyone else can immediately see the latest version. This way, no one works with outdated information, and the final output is based on a complete, consistent picture of the task.

**Technical**: The `ChatbotState` (defined in **Cell 16**) is a `TypedDict` that acts as a mutable, shared state object. It's explicitly designed to hold all relevant information pertaining to a single conversation turn, or an ongoing conversation thread, including:

-   `user_id`, `session_id`, `user_message`: Core identifiers and the current user input.
-   `memory`: Retrieved conversation history and lead status.
-   `route`: The intent and tools identified by the Orchestrator.
-   `sqlite_sql`, `sqlite_result`: SQL query generated and its execution result.
-   `duckdb_sql`, `duckdb_result`: DuckDB query generated and its execution result.
-   `chroma_result`: Context retrieved from ChromaDB.
-   `recommendation`: The output from the Recommendation Agent.
-   `final_response`: The ultimate response crafted by the Response Agent.

Here's how it ensures consistency:

1.  **Shared Mutable Object**: The `ChatbotState` is a single instance of a dictionary-like object that is passed from one agent to the next within the `LangGraph` workflow (`Cell 24`). Each agent receives the *current* state, processes it, adds or modifies relevant fields, and then passes the *updated* state to the next agent.

2.  **Sequential Updates**: Because `LangGraph` (`Cell 24`) enforces a defined flow (e.g., `memory` -> `orchestrator` -> `sql` -> ... -> `response`), each agent operates on the state that has already been processed and enriched by all preceding agents in that specific path. For example:
    *   The `memory_agent` (`Cell 17`) updates `state['memory']`.
    *   The `master_orchestrator_agent` (`Cell 18`) then receives this `state` (which now includes `memory`) and updates `state['route']`.
    *   The `sql_agent` (`Cell 19`) then receives the `state` (which includes `memory` and `route`) and updates `state['sqlite_sql']` and `state['sqlite_result']`.
    
    This chained update ensures that every agent always works with the most up-to-date and complete picture of the conversation and any information gathered so far.

3.  **Explicit Data Fields**: The `TypedDict` definition (`Cell 16`) provides a clear structure, making it easy for developers to understand what information is available in the state and where each agent's output is expected to be stored. This reduces ambiguity and helps prevent agents from inadvertently overwriting or misinterpreting data.

**Real-Time Scenario**: Consider the user query: "I am a beginner and I want to learn AI. Which course should I start with?"

-   **Initial State**: `{'user_message': '...', 'user_id': '...', 'session_id': '...'}`
-   **`memory_agent` Updates**: Adds `memory` (e.g., `{'recent_history': [...], 'lead_status': [{'status': 'new', 'score': 20}]}`) to `ChatbotState`.
-   **`master_orchestrator_agent` Updates**: Based on the `user_message` and `memory`, it adds `route` (e.g., `{'intent': 'course_lookup', 'tools_needed': ['sqlite', 'chroma'], ...}`) to `ChatbotState`.
-   **`sql_agent` Updates**: Based on `user_message`, `memory`, and `route`, it adds `sqlite_sql` (e.g., `SELECT ... FROM courses ...`) and `sqlite_result` (e.g., `[{'course_id': 'C001', 'course_name': 'Python for AI Foundations'}]`) to `ChatbotState`.
-   **`rag_agent` Updates**: Searches ChromaDB using the `user_message` and adds `chroma_result` (e.g., `['For beginners, Python for AI Foundations is recommended...']`) to `ChatbotState`.
-   **`recommendation_agent` Updates**: Takes *all* the above fields (`user_message`, `memory`, `route`, `sqlite_result`, `chroma_result`) and adds `recommendation` (e.g., 'Python for AI Foundations is a good starting point...') to `ChatbotState`.
-   **`response_agent` Updates**: Takes the complete `ChatbotState` and generates `final_response` (e.g., 'To start your journey...'), also saving it to `memory` (which is again written to the database via MCP).

At every step, the agents have access to the cumulative, consistent information. This prevents agents from making decisions based on incomplete or outdated data, ensuring a smooth and accurate conversational flow.